# Disney Exploration (Surgically Adapted)

Adapted from `spark_enrichment.ipynb`. Enrichment actions removed. Focus is on data exploration.

In [ ]:
import sys
from pathlib import Path
# Add src to sys.path so we can import stampli package
notebook_dir = Path("__file__").parent.resolve() if "__file__" in locals() else Path(".").resolve()
src_path = str(notebook_dir.parents[0]) # src/stampli -> src
if src_path not in sys.path:
    sys.path.append(src_path)
print(f"Added {src_path} to sys.path")


In [1]:
from stampli.paths import get_enriched_path
ENRICHED_PATH = str(get_enriched_path())


In [2]:
import pandas as pd
import os
import importlib
from pyspark.sql.functions import col, lower

import stampli.util.display
importlib.reload(stampli.util.display)
from stampli.util.display import display_scrollable_dataframe

In [3]:
# 1. Setup Spark
try:
    from pyspark.sql import SparkSession
    from stampli.util.runtime import bootstrap_spark_env
except ImportError:
    print("Please install pyspark: pip install pyspark")

bootstrap_spark_env()

spark = (SparkSession.builder
    .appName("DisneyExploration")
    .config("spark.executor.memory", "16g")
    .config("spark.driver.memory", "16g")
    .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")
print("Spark Session Created (16GB)")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/18 09:46:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Session Created (16GB)


In [4]:
# 3. Load Prior Enrichment (The Data)
if os.path.exists(ENRICHED_PATH) and len(os.listdir(ENRICHED_PATH)) > 0:
    try: 
        sdf_enriched = spark.read.parquet(ENRICHED_PATH)
        total_count = sdf_enriched.count()
        print(f"Total Enriched Rows: {total_count}")
    except Exception as e:
        print(f"Error reading data: {e}")
        sdf_enriched = None
else:
    print("No enrichment data found at path.")
    sdf_enriched = None

Total Enriched Rows: 42656


In [5]:
# 4. Filter for Disney
# Creating spd_disney (Spark DataFrame)

# Broad filter for 'Disney' in Branch
spd_disney = sdf_enriched.filter(lower(col("Branch")).contains("disney"))

disney_count = spd_disney.count()
print(f"Disney Rows Found: {disney_count}")

# Show branch breakdown
print("Branches present:")
spd_disney.select("Branch").distinct().show(truncate=False)

Disney Rows Found: 42656
Branches present:


+---------------------+
|Branch               |
+---------------------+
|Disneyland_HongKong  |
|Disneyland_Paris     |
|Disneyland_California|
+---------------------+



In [6]:
# 5. Create Pandas DataFrame (pd_disney)

print("Converting to Pandas (pd_disney)...")
pd_disney = spd_disney.toPandas()

print(f"Pandas DF Shape: {pd_disney.shape}")
display_scrollable_dataframe(pd_disney)

Converting to Pandas (pd_disney)...


Pandas DF Shape: (42656, 18)


## Verification Blocks

In [7]:
# Check for 'Crowded' / 'Packed' reviews in specific locations
print("Crowd Level Distribution:")
print(pd_disney.groupby('Branch')['crowd_level'].value_counts())

Crowd Level Distribution:
Branch               crowd_level
Disneyland_HongKong  Moderate       187
                     Packed          67
                     Empty           59
Disneyland_Paris     Moderate       719
                     Packed         381
                     Empty           73
Name: count, dtype: int64


In [10]:
# Keyword Check vs Extraction
KEYWORDS = ["crowd", "packed", "busy", "line", "queue", "wait", "full", "people"]

# Filter for reviews with keywords
mask = pd_disney['Review_Text'].str.contains('|'.join(KEYWORDS), case=False, na=False)
subset = pd_disney[mask]

print(f"Reviews with crowd keywords: {len(subset)}")

print("Of which have extracted crowd_level:")
print(subset['crowd_level'].value_counts(dropna=False))

# Show mis-matches (Keyword present, but crowd_level is NaN)
missed = subset[subset['crowd_level'].isna()]
if not missed.empty:
    print(f"\nPotential Misses ({len(missed)}):")
    display_scrollable_dataframe(missed[['Branch', 'Review_Text', 'summary']])

Reviews with crowd keywords: 26699
Of which have extracted crowd_level:
crowd_level
None        25324
Moderate      815
Packed        437
Empty         123
Name: count, dtype: int64

Potential Misses (25324):


In [9]:
# Specific Investigation: California in June
# User Question: "Is Disneyland California usually crowded in June?"

# 1. Parse Month
def get_month(ym):
    try:
        return int(ym.split('-')[1])
    except:
        return 0

pd_disney['month'] = pd_disney['Year_Month'].apply(get_month)

# 2. Filter CA + June
ca_june = pd_disney[
    (pd_disney['Branch'] == 'Disneyland_California') & 
    (pd_disney['month'] == 6)
]

print(f"Total Reviews (CA + June): {len(ca_june)}")
print("\nExtracted Crowd Levels:")
print(ca_june['crowd_level'].value_counts(dropna=False))

# 3. False Negative Check
KEYWORDS = ["crowd", "packed", "busy", "line", "queue", "wait"]
mask_keys = ca_june['Review_Text'].str.contains('|'.join(KEYWORDS), case=False, na=False)
missed_ca_june = ca_june[mask_keys & ca_june['crowd_level'].isna()]

print(f"\nPotential False Negatives (Keywords present, Label Missing): {len(missed_ca_june)}")
if not missed_ca_june.empty:
    display_scrollable_dataframe(missed_ca_june[['Review_Text', 'summary']])

Total Reviews (CA + June): 1647

Extracted Crowd Levels:
crowd_level
None    1647
Name: count, dtype: int64

Potential False Negatives (Keywords present, Label Missing): 947


In [ ]:
yo = """
We visited Disneyland with our 9 and 7 year old children in June 2016. 
We bought a 3 day park hopper, and stayed in the nearby Fairfield Inn (also excellent). 
Disneyland was amazing, which is even more impressive given that I'm not a theme park person (the visit was for my wife and kids, 
not me) and the fact that we had ridiculously high expectations. Things that made our visit great: 
we used fastpasses extensively 
(learn how these work and use them to avoid the lines with the exception of 2 3 lines we didn't queue anywhere for more than 15 minutes) 
we booked dinners in the parks so we had reserved viewing areas for the shows (Paint the Night and World of Colour) 
Frozen at the Hyperion staying across the road so we could give the kids an hour of downtime each day 
(we went from park open to park close everyday) 
the characters were amazing, accessible, 
and very enthusiastic Goofy's kitchen General comments everything was clean and comfortable queues were enjoyable; 
they moved quickly, they had entertainment ride build up staff were accomodatingFavourite rides (DisneyLand)
 Splash Mountain Buzz Lightyear Astro Blasters Matterhorn Bobsleds Indiana Jones Jungle Cruise (awesome captain) 
 Space Mountain Big Thunder Mountain Star Wars The Adventure Continues (split jury on this one)
 Finding Nemo Pirates of the CarribeanFavourite rides (California Adventureland) 
 California Screamin' Grizzly River Run Tower of Terror (split decision) Radiator Springs Racing 
 (the fastpasses run out really quickly get one before 10am) Toy Story Midway Mania Soaring over CaliforniaOnly negatives 
 (pretty minor): no wifi in the park despite internet access being essential to planning (get the app to see real time queue times). 
 no stamps available in the park, 
 despite the emporium selling postcards and having a postbox immediately outside downtown dining is crap, 
 and has really long wait times (don't even bother trying to walk up without a reservation)Overall an amazing experience, 
 a must do family trip.	
"""